In [ ]:
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm
import time
from scipy import linalg
import porespy as ps
from skimage import draw
import pickle
import os
import glob
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
import sys
import os
sys.path.append(os.path.abspath('..'))
from source.features2D import (
    road_graph_to_binary_image as graph_to_binary_image,
    get_edge_coords, compute_branches, 
    average_bending_energy_curve, 
    compute_fractal_dimension_road as compute_fractal_dimension
)

In [ ]:
csv_data = []

# Function to create a 2D binary image from road network

pickle_folder = '../pkl/pkl_city_graphs'

pickle_files = glob.glob(os.path.join(pickle_folder, '*.pkl'))
for pickle_file in tqdm(pickle_files):
    print(f"\nProcessing pickle file: {pickle_file}")
  
    try:
        with open(pickle_file, 'rb') as f:
            combined_graph = pickle.load(f)
      
        # Extract name and type from graph attributes
        name = combined_graph.graph.get('name', os.path.basename(pickle_file).replace('.pkl', '').replace('_', ' '))
        region_type = combined_graph.graph.get('type', 'unknown')
      
        print(f"Loaded graph for {name} (type: {region_type})")
    except Exception as e:
        print(f"Error loading {pickle_file}: {e}")
        continue
  
    # Compute branches
    branch_lengths, branch_polylines = compute_branches(combined_graph)
    number_of_egdes = len(branch_lengths)
    avg_branch_length = np.mean(branch_lengths) if branch_lengths.size > 0 else 0
    
    # Compute edge directions and weights
    local_vectors = []
    local_lengths = []
    straight_line_lengths = []
    all_points = [] 


    for u, v, data in combined_graph.edges(data=True):
        if 'geometry' in data:
            xs, ys = data['geometry'].xy
            coords = np.array(list(zip(xs, ys)))
        else:
            x1, y1 = combined_graph.nodes[u]['x'], combined_graph.nodes[u]['y']
            x2, y2 = combined_graph.nodes[v]['x'], combined_graph.nodes[v]['y']
            coords = np.array([[x1, y1], [x2, y2]])

        # accumulate points for convex hull
        all_points.extend(coords.tolist())  # <- safer: convert array rows to list

        total_edge_length = 0
        for i in range(len(coords) - 1):
            x0, y0 = coords[i]
            x1, y1 = coords[i + 1]
            dx, dy = x1 - x0, y1 - y0
            length = np.sqrt(dx**2 + dy**2)
            if length > 0:
                local_vectors.append((dx / length, dy / length))
                local_lengths.append(length)
                total_edge_length += length

        # Straight-line distance between start and end nodes
        straight_distance = np.sqrt((coords[-1, 0] - coords[0, 0])**2 +
                                    (coords[-1, 1] - coords[0, 1])**2)
        straight_line_lengths.append(straight_distance)

    # Skip if no valid segments
    if len(local_vectors) == 0:
        print(f"No valid segments for {name}")
        continue

    # ---- edge density: length / area ----
    all_points = np.array(all_points)
    if len(np.unique(all_points, axis=0)) >= 3:
        hull = ConvexHull(all_points)
        area = hull.volume  # in 2D this is area
    else:
        area = 0
        print(f"Warning: Not enough unique points to compute convex hull for {name}. Setting area to 0.")

    total_path_length = sum(local_lengths)
    branch_density = total_path_length / area if area > 0 else 0
    # -----------------------------------------


    
    try:
        bc = nx.betweenness_centrality(combined_graph, weight='length', normalized=True)


        
        mean_betweenness_centrality = np.mean(list(bc.values())) if bc else 0
    except:
        mean_betweenness_centrality = 0
    try:
        graph_diameter = nx.diameter(combined_graph, weight='length') if nx.is_connected(combined_graph) else 0
    except:
        graph_diameter = 0
    try:
        average_path_length = nx.average_shortest_path_length(combined_graph, weight='length') if nx.is_connected(combined_graph) else 0
    except:
        average_path_length = 0
    try:
        assortativity = nx.degree_assortativity_coefficient(combined_graph, weight='length') if combined_graph.number_of_edges() >= 2 else 0
        assortativity = 0 if np.isnan(assortativity) else assortativity
    except:
        assortativity = 0
    try:
        L = nx.laplacian_matrix(combined_graph, weight='length').astype(float).toarray()
        eigenvalues = linalg.eigvalsh(L)
        eigenvalues = np.maximum(eigenvalues, 1e-12)
        probs = eigenvalues / np.sum(eigenvalues)
        spectral_entropy = -np.sum(probs * np.log(probs)) if probs.sum() > 0 else 0
    except:
        spectral_entropy = 0
      
    algebraic_connectivity = nx.algebraic_connectivity(combined_graph, weight='length', method='tracemin_pcg') if nx.is_connected(combined_graph) else 0
  
    # Compute circuity
    total_path_length = sum(local_lengths)
    total_straight_length = sum(straight_line_lengths) if straight_line_lengths else 0
    circuity = total_path_length / total_straight_length if total_straight_length > 0 else 1.0
    
    # Compute angular metrics
    angles = np.arctan2([dy for dx, dy in local_vectors], [dx for dx, dy in local_vectors]) % np.pi
    angles_full = 2 * angles
    edge_weights = np.array(local_lengths) / np.sum(local_lengths)
    C_p = np.sum(edge_weights * np.cos(angles_full))
    S_p = np.sum(edge_weights * np.sin(angles_full))
    theta_mean_full = np.arctan2(S_p, C_p) % (2 * np.pi)
  
    theta_mean = theta_mean_full / 2
    R_p_mean = np.sqrt(C_p**2 + S_p**2)
  
    directional_std_dev = np.sqrt(-2 * np.log(R_p_mean)) if R_p_mean > 0 else 0
    rotated_angles = (angles - theta_mean + np.pi/2) % np.pi
  
    mask_4 = ((rotated_angles >= 0) & (rotated_angles <= np.pi/8)) | ((rotated_angles >= 7*np.pi/8) & (rotated_angles <= np.pi))
    mask_3 = ((rotated_angles > np.pi/8) & (rotated_angles <= np.pi/4)) | ((rotated_angles >= 3*np.pi/4) & (rotated_angles < 7*np.pi/8))
    mask_2 = ((rotated_angles > np.pi/4) & (rotated_angles <= 3*np.pi/8)) | ((rotated_angles >= 5*np.pi/8) & (rotated_angles < 3*np.pi/4))
    mask_1 = (rotated_angles > 3*np.pi/8) & (rotated_angles < 5*np.pi/8)
  
    mass_quartile_1 = np.sum(edge_weights * mask_1)
    mass_quartile_2 = np.sum(edge_weights * mask_2)
    mass_quartile_3 = np.sum(edge_weights * mask_3)
    mass_quartile_4 = np.sum(edge_weights * mask_4)
  
    n_bins = 18
    bin_edges = np.linspace(0, np.pi, n_bins + 1)
    hist_weighted, _ = np.histogram(rotated_angles, bins=bin_edges, weights=edge_weights)
    p_weighted = hist_weighted / np.sum(hist_weighted) if np.sum(hist_weighted) > 0 else np.zeros_like(hist_weighted)
    directional_entropy = -np.sum([p * np.log(p) if p > 0 else 0 for p in p_weighted])
    orientation_order = 1 - (directional_entropy / np.log(18))**2 if directional_entropy > 0 else 0
    # Compute fractal dimension
    fractal_dimension = compute_fractal_dimension(combined_graph)
  
    # Compute average bending energy
    branch_avg_bendings = []
    for c in branch_polylines:
        if c.shape[0] >= 3:
            try:
                avg_bend = average_bending_energy_curve(c, closed=False)
                branch_avg_bendings.append(avg_bend)
            except ValueError as e:
                print(f"Skipping branch in {name}: {e}")
    bending_energy = sum(branch_avg_bendings) if branch_avg_bendings else 0.0
  
 # Prepare data for CSV
    csv_row = {
        'region_name': name,
        'type': region_type,
        
        
        'N_edg': number_of_egdes,           
        'Betw': mean_betweenness_centrality,
        'Spec_ent': spectral_entropy,
        'Alg_Conn': algebraic_connectivity,
        'Assort': assortativity,
        'G_diam': graph_diameter,
        'Br_dens': branch_density,           
        'Br_l': avg_branch_length,
        'Energy': bending_energy,
        'Apl': average_path_length,
        'Circ': circuity,
        'FD': fractal_dimension,
        'Dir_std': directional_std_dev,
        'q1': mass_quartile_1,
        'q2': mass_quartile_2,
        'q3': mass_quartile_3,
        'q4': mass_quartile_4,
        'Dir_ent': directional_entropy,
        'Order': orientation_order
    }
    csv_data.append(csv_row)
   

    
# Save features to CSV
csv_columns = [
    'region_name', 'type',
    'N_edg',
    'Betw',
    'Spec_ent',
    'Alg_Conn',
    'Assort',
    'G_diam',
    'Br_dens',
    'Br_l',
    'Energy',
    'Apl',
    'Circ',
    'FD',
    'Dir_std',
    'q1',
    'q2',
    'q3',
    'q4',
    'Dir_ent',
    'Order'
]

df = pd.DataFrame(csv_data, columns=csv_columns)
df.to_csv('road_network_features.csv', index=False)
print("\nFeature data saved to 'road_network_features.csv'")